A.


In [ ]:
import numpy as np

# ----- Define A -----
A = (1/10) * np.array([
    [ 5, -1, -2,  0],
    [-1,  5,  0, -2],
    [-2,  0,  5, -1],
    [ 0, -2, -1,  5]
], dtype=float)

# ----- Define b -----
b = np.array([1, 0, 0, 0], dtype=float)

# ----- Compute eigenvalues & eigenvectors -----
# np.linalg.eigh returns eigenvalues in ascending order
eigvals, eigvecs = np.linalg.eigh(A)

# ----- Factor out 0.2 and express eigenvalues as nice fractions -----
# We know 0.2 = 1/5, so eigenvalues should be k/5 for k = 1,2,3,4
scaled_eigvals = np.round(eigvals / 0.2).astype(int)  # k ≈ 1,2,3,4

print("Eigenvalues of A (as fractions k/5):")
for i, k in enumerate(scaled_eigvals, start=1):
    frac_str = f"{k}/5"
    print(f"λ{i} = {frac_str}  (numeric ≈ {eigvals[i-1]:.6f})")

# ----- Print eigenvector matrix -----
print("\nEigenvector matrix (columns are eigenvectors):")
print(eigvecs)

# ----- Sanity checks on eigenvectors -----
print("\nCheck: ||A v_i - λ_i v_i|| for each eigenpair:")
for i, (lam, v) in enumerate(zip(eigvals, eigvecs.T), start=1):
    residual = np.linalg.norm(A @ v - lam * v)
    print(f"i = {i}: residual ≈ {residual:.3e}")

# Check orthonormality of eigenvectors: V^T V should be ~ I
VtV = eigvecs.T @ eigvecs
print("\nV^T V (should be close to identity):")
print(VtV)


B.


In [ ]:
import numpy as np

# This cell assumes:
#   - b is your 4D real vector (shape: (4,))
#   - eigvecs is the 4x4 eigenvector matrix from np.linalg.eigh(A)
#     (columns are the eigenvectors |u_i>)

# 1. Normalize b → |b>
norm_b = np.linalg.norm(b)
ket_b = b / norm_b

print("Normalized state |b> (in computational basis):")
print(ket_b)
print()

# 2. Expansion coefficients c_i = ⟨u_i|b⟩
#    Columns of eigvecs are |u_i>, so eigvecs.T.conj() @ |b> gives all c_i
c = eigvecs.T.conj() @ ket_b

print("|b> expanded in the eigenbasis {|u_i>} of A:")
print("    |b> = Σ_i c_i |u_i>")
print("Coefficients c_i (⟨u_i|b⟩):")
for i, ci in enumerate(c, start=1):
    # For clarity, separate real/imag if needed
    if abs(ci.imag) < 1e-10:
        # Purely real (or almost) → print as real
        print(f"  c_{i} = {ci.real:.6f}")
    else:
        # Complex → a + ib format
        print(f"  c_{i} = {ci.real:.6f} + {ci.imag:.6f} i")
print()

# 3. Check normalization in eigenbasis: Σ |c_i|^2 should be 1
prob_sum = np.sum(np.abs(c)**2)
print("Sum of probabilities Σ_i |c_i|^2 =", f"{prob_sum:.6f}")
print()

# 4. Reconstruct |b> from the eigenbasis to verify consistency
reconstructed_b = eigvecs @ c

print("Reconstructed |b> from Σ_i c_i |u_i>:")
print(reconstructed_b)
print()

diff_norm = np.linalg.norm(reconstructed_b - ket_b)
print("‖ |b> - reconstructed |b> ‖ =", f"{diff_norm:.3e}")


C.

In [ ]:
import numpy as np

# This cell assumes A and b are already defined as:
#   - A: 4x4 matrix
#   - b: 4D vector
# If not, uncomment and run the block below.
"""
A = (1/10) * np.array([
    [ 5, -1, -2,  0],
    [-1,  5,  0, -2],
    [-2,  0,  5, -1],
    [ 0, -2, -1,  5]
], dtype=float)

b = np.array([1, 0, 0, 0], dtype=float)
"""

# 1) Solve A x = b classically
x = np.linalg.solve(A, b)

print("Classical solution to A x = b:")
print("x =", x)

# 2) Normalize x to interpret it as a quantum state |x⟩
norm_x = np.linalg.norm(x)
ket_x = x / norm_x

print("\nNorm of x: ‖x‖ =", f"{norm_x:.6f}")

print("\nNormalized solution |x⟩ in the computational basis (2 qubits):")
print("  |x⟩ = Σ_i α_i |i⟩,  where i = 00, 01, 10, 11")
for i, amp in enumerate(ket_x):
    basis_label = f"{i:02b}"  # 0 -> '00', 1 -> '01', etc.
    print(f"  α_{basis_label} = {amp:.6f}")


HHL - implementation 


In [ ]:
# ==============================
# Cell 1 – Dependencies & Setup
# ==============================


import numpy as np

from qiskit import QuantumCircuit, QuantumRegister, ClassicalRegister
from qiskit_aer import AerSimulator

from qiskit.circuit.library import UnitaryGate





# Optional: nicer NumPy printing
np.set_printoptions(precision=6, suppress=True)

# ----- Define A -----
A = (1/10) * np.array([
    [ 5, -1, -2,  0],
    [-1,  5,  0, -2],
    [-2,  0,  5, -1],
    [ 0, -2, -1,  5]
], dtype=float)

# ----- Define b -----
b = np.array([1, 0, 0, 0], dtype=float)

print("Matrix A:")
print(A)
print("\nVector b:")
print(b)

# ----- Classical solution: A x = b -----
x = np.linalg.solve(A, b)

print("\nClassical solution x to A x = b:")
print("x =", x)

# Normalize x to interpret it as |x⟩
norm_x = np.linalg.norm(x)
ket_x = x / norm_x

print("\nNorm of x: ‖x‖ =", f"{norm_x:.6f}")
print("\nNormalized solution |x⟩ in computational basis (2 qubits):")
print("  |x⟩ = Σ_i α_i |i⟩,  where i ∈ {00, 01, 10, 11}")
for i, amp in enumerate(ket_x):
    basis_label = f"{i:02b}"
    print(f"  α_{basis_label} = {amp:.6f}")

# Backend for later simulation
simulator = AerSimulator()

print("\n[Setup complete] Ready to define registers, QFT/QPE blocks, and HHL circuit in the next cell.")


HHL circuit

In [ ]:
# ==============================
# Cell 2 – QFT/QPE, calibrated R_y, HHL circuit
# ==============================

from qiskit import QuantumCircuit, QuantumRegister, ClassicalRegister, transpile

# --- 0. Eigen-decomposition of A (used for U = e^{iAt}) ---

eigvals, eigvecs = np.linalg.eigh(A)   # ascending order

print("Eigenvalues λ_i:")
for i, lam in enumerate(eigvals, start=1):
    print(f"  λ_{i} = {lam:.6f}")

# --- 1. Registers and main circuit ---

ancilla = QuantumRegister(1, "anc")
clock   = QuantumRegister(3, "clock")   # phase register
b_reg   = QuantumRegister(2, "b")       # system register
creg    = ClassicalRegister(3, "c")     # c[0]=anc, c[1:3]=system bits

circuit = QuantumCircuit(ancilla, clock, b_reg, creg)

# --- 2. QFT and inverse QFT on 3 clock qubits ---

def qft_dagger(circ, q):
    """Inverse QFT on 3-qubit clock register q[0], q[1], q[2]."""
    circ.h(q[2])
    circ.cp(-np.pi / 2, q[1], q[2])
    circ.cp(-np.pi / 4, q[0], q[2])
    circ.h(q[1])
    circ.cp(-np.pi / 2, q[0], q[1])
    circ.h(q[0])
    circ.swap(q[0], q[2])

def qft(circ, q):
    """QFT on 3-qubit clock register q[0], q[1], q[2]."""
    circ.swap(q[0], q[2])
    circ.h(q[0])
    circ.cp(np.pi / 2, q[0], q[1])
    circ.h(q[1])
    circ.cp(np.pi / 4, q[0], q[2])
    circ.cp(np.pi / 2, q[1], q[2])
    circ.h(q[2])

# --- 3. Controlled U^powers based on A, eigvals, eigvecs ---

t = 5 * np.pi / 4  # evolution time used for U = exp(i A t)

def controlled_U_power(circ, clock_qubit, system_reg, power, sign=+1):
    """
    Apply controlled-U^power, where U = exp(i * sign * A * t),
    using A = V diag(λ) V†, with eigvals, eigvecs from np.linalg.eigh(A).
    """
    phases = np.exp(1j * sign * eigvals * t * power)
    U = eigvecs @ np.diag(phases) @ eigvecs.conj().T
    U_gate = UnitaryGate(U, label=f"U^{power}" if sign > 0 else f"U^-{power}")
    circ.append(U_gate.control(1), [clock_qubit, system_reg[0], system_reg[1]])

def apply_controlled_U_powers(circ, clock_reg, system_reg, sign=+1):
    """Apply U^(2^k) with k=0,1,2 on clock[0],clock[1],clock[2]."""
    controlled_U_power(circ, clock_reg[0], system_reg, power=1, sign=sign)
    controlled_U_power(circ, clock_reg[1], system_reg, power=2, sign=sign)
    controlled_U_power(circ, clock_reg[2], system_reg, power=4, sign=sign)

# --- 4. QPE and inverse QPE blocks ---

def qpe(circ, clock_reg, system_reg):
    """QPE: H on clock, controlled U^(2^k), inverse QFT."""
    circ.barrier(label="QPE: H on clock")
    circ.h(clock_reg)
    circ.barrier(label="QPE: CU^(2^k)")
    apply_controlled_U_powers(circ, clock_reg, system_reg, sign=+1)
    circ.barrier(label="QPE: inverse QFT")
    qft_dagger(circ, clock_reg)

def inv_qpe(circ, clock_reg, system_reg):
    """Inverse QPE: QFT + U^(2^k)†."""
    circ.barrier(label="inv QPE: QFT")
    qft(circ, clock_reg)
    circ.barrier(label="inv QPE: CU^(2^k)†")
    apply_controlled_U_powers(circ, clock_reg, system_reg, sign=-1)

# --- 5. Calibrate pattern → eigenvalue using QPE alone ---

def qpe_on_eigenvector(eig_index, shots=2048):
    """
    Run QPE on eigenvector |u_i> and return dominant clock pattern.
    Bitstring is c2 c1 c0 (from Qiskit’s classical register order).
    """
    clk  = QuantumRegister(3, "clk")
    btmp = QuantumRegister(2, "btmp")
    creg = ClassicalRegister(3, "c")

    qc = QuantumCircuit(clk, btmp, creg)

    # prepare |u_i> on btmp
    u = eigvecs[:, eig_index]
    qc.initialize(u, btmp)

    # QPE
    qc.h(clk)
    apply_controlled_U_powers(qc, clk, btmp, sign=+1)
    qft_dagger(qc, clk)

    # measure clock
    qc.measure(clk[0], creg[0])  # -> c0
    qc.measure(clk[1], creg[1])  # -> c1
    qc.measure(clk[2], creg[2])  # -> c2

    tc = transpile(qc, simulator)
    result = simulator.run(tc, shots=shots).result()
    counts = result.get_counts()

    dominant = max(counts, key=counts.get)  # bitstring "c2 c1 c0"
    return dominant

pattern_to_lambda = {}
print("\n=== Calibrating QPE pattern→eigenvalue mapping ===")
for i, lam in enumerate(eigvals):
    pat = qpe_on_eigenvector(i)
    pattern_to_lambda[pat] = lam
    print(f"  eigenvalue λ_{i+1} ≈ {lam:.6f}  →  clock pattern {pat}")

print("\npattern_to_lambda =", pattern_to_lambda)

# --- 6. Multi-eigenvalue controlled R_y(θ(λ)) using calibrated patterns ---

def apply_eigenvalue_controlled_ry(circ, clock_reg, anc_reg, pattern_to_lambda, C=None):
    """
    For each clock pattern p in pattern_to_lambda:
      λ = pattern_to_lambda[p]
      θ = 2 arcsin(C/λ)
    apply R_y(θ) to ancilla, controlled on (clock == p).

    Here p is a string "c2c1c0" corresponding to
      c2 -> clock[2], c1 -> clock[1], c0 -> clock[0].
    """
    if C is None:
        C = min(pattern_to_lambda.values())

    for pattern, lam in pattern_to_lambda.items():
        ratio = C / lam
        if ratio > 1.0:
            raise ValueError(f"C/λ = {ratio} > 1 for λ={lam}, choose smaller C.")

        theta = 2 * np.arcsin(ratio)

        controls = []
        flipped = []

        # pattern[0] = bit for clock[2], pattern[1] for clock[1], pattern[2] for clock[0]
        bit_to_qubit = {0: 2, 1: 1, 2: 0}

        for bit_pos, bit in enumerate(pattern):
            q_index = bit_to_qubit[bit_pos]
            if bit == "0":
                circ.x(clock_reg[q_index])
                flipped.append(q_index)
            controls.append(clock_reg[q_index])

        # multi-controlled R_y(θ) on ancilla
        circ.mcry(theta, controls, anc_reg[0], None)

        # uncompute X flips
        for q_index in flipped:
            circ.x(clock_reg[q_index])

# --- 7. HHL macro using calibrated rotation angles ---

def hhl(circ, anc_reg, clock_reg, system_reg, classical_reg):
    """
    HHL:
      1) QPE (on clock + system)
      2) Eigenvalue-dependent R_y(θ(λ)) on ancilla
      3) Measure ancilla (post-select anc=1 in analysis)
      4) Inverse QPE
    """
    circ.barrier(label="QPE")
    qpe(circ, clock_reg, system_reg)

    circ.barrier(label="controlled R_y (eigenvalue-dependent)")
    apply_eigenvalue_controlled_ry(
        circ,
        clock_reg,
        anc_reg,
        pattern_to_lambda,
        C=min(eigvals)
    )

    circ.barrier(label="measure ancilla")
    circ.measure(anc_reg[0], classical_reg[0])

    circ.barrier(label="inverse QPE")
    inv_qpe(circ, clock_reg, system_reg)

# --- 8. State preparation for |b> and full HHL circuit ---

circuit.barrier(label="prepare |b>")
# |b> = |00> already on b_reg, so nothing to do

circuit.barrier(label="run HHL")
hhl(circuit, ancilla, clock, b_reg, creg)

circuit.barrier(label="measure system")
circuit.measure(b_reg[0], creg[1])
circuit.measure(b_reg[1], creg[2])

# --- 9. Draw circuit ---

circuit.draw(output="mpl")


In [ ]:
# ==============================
# Cell 4 – Simulate HHL, post-select ancilla=1, estimate |x>
# ==============================

from qiskit_aer import AerSimulator
from qiskit import transpile
import numpy as np

sim = AerSimulator()

# Transpile circuit for simulator backend
tcirc = transpile(circuit, sim)

# Run simulation
result = sim.run(tcirc, shots=20000).result()
counts = result.get_counts()

print("Raw measurement counts:")
print(counts)

# --------------------------------------------------------
# POST-SELECTION: keep only outcomes with ancilla = 1
# --------------------------------------------------------
post_counts = {}
total_post = 0

for bitstring, count in counts.items():
    # Classical bits are c2 c1 c0, with c0 = ancilla (measured into creg[0])
    ancilla_bit = bitstring[-1]      # c0 (rightmost)
    if ancilla_bit == '1':
        post_counts[bitstring] = count
        total_post += count

print("\nPost-selected (ancilla=1) counts:")
print(post_counts)
print(f"Total successful shots: {total_post}")

# --------------------------------------------------------
# Extract approximate |x⟩ from system bits
# --------------------------------------------------------
amps = np.zeros(4)

for bitstring, count in post_counts.items():
    # bitstring = c2 c1 c0, with c1,c2 = system bits (b1,b0)
    sys_bits = bitstring[-3:-1]   # c2 c1
    idx = int(sys_bits, 2)
    amps[idx] += count

# Convert counts → probabilities → amplitudes
amps = amps / np.sum(amps)
amps = np.sqrt(amps)

print("\nEstimated |x⟩ from HHL (normalized):")
for i in range(4):
    print(f"  |{i:02b}> amplitude ≈ {amps[i]:.6f}")

print("\nClassical |x⟩ (from Cell 1):")
print("  |00> =", ket_x[0])
print("  |01> =", ket_x[1])
print("  |10> =", ket_x[2])
print("  |11> =", ket_x[3])


In [ ]:
from qiskit.visualization import plot_histogram

plot_histogram(counts)


Variational algorithm

In [ ]:
# ==============================
# Cell 3a – Build H and get Pauli decomposition (new API)
# ==============================

import numpy as np
from qiskit.quantum_info import SparsePauliOp, Operator

# We assume A, b, ket_x are already defined from previous cells.

# 1) Define |b><b| projector
ket_b = b / np.linalg.norm(b)          # already [1,0,0,0] here
proj_b = np.outer(ket_b, ket_b.conj()) # |b><b|

# 2) Identity on 4D space
I4 = np.eye(4)

# 3) Hamiltonian H = A (I - |b><b|) A
H_mat = A @ (I4 - proj_b) @ A

print("Matrix H = A (I - |b><b|) A :")
print(H_mat)

# 4) Check that |x> is a zero-eigenvector: H|x> ≈ 0
residual = np.linalg.norm(H_mat @ ket_x)
print("\n‖H |x⟩‖ (should be ~0):", residual)

# 5) Convert H into a sum of Paulis using SparsePauliOp
H_pauli = SparsePauliOp.from_operator(Operator(H_mat))

print("\nPauli decomposition of H (SparsePauliOp):")
print(H_pauli)


In [ ]:
# ==============================
# Cell 3b – VQE to find ground state of H (compatible version)
# ==============================

import os
import numpy as np
NOTEBOOK_CI = os.environ.get("NOTEBOOK_CI") == "1"
from qiskit.circuit.library import EfficientSU2
from qiskit_algorithms import VQE
from qiskit.quantum_info import Statevector
from qiskit_algorithms.optimizers import SLSQP
from qiskit.primitives import StatevectorEstimator

estimator = StatevectorEstimator()

# 2-qubit ansatz
ansatz = EfficientSU2(num_qubits=2, reps=2, entanglement="full")

optimizer = SLSQP(maxiter=5 if NOTEBOOK_CI else 200)

vqe = VQE(estimator=estimator, ansatz=ansatz, optimizer=optimizer)

vqe_result = vqe.compute_minimum_eigenvalue(H_pauli)

print("VQE ground state energy:", vqe_result.eigenvalue.real)

# Extract eigenstate vector
psi_vqe = np.array(Statevector.from_instruction(
    ansatz.assign_parameters(vqe_result.optimal_parameters)
).data).flatten()

print("\nVQE ground-state amplitudes:")
for i, amp in enumerate(psi_vqe):
    print(f"  |{i:02b}>: {amp:.6f}")

# Compare with classical |x>
overlap = np.vdot(ket_x, psi_vqe)
fidelity = np.abs(overlap)**2
print("\n|<x_classical | x_VQE>|^2 =", fidelity)

# Remove global phase for comparison
phase = overlap / np.abs(overlap)
psi_aligned = psi_vqe * np.conj(phase)

print("\nClassical |x> amplitudes:")
for i, amp in enumerate(ket_x):
    print(f"  |{i:02b}>: {amp:.6f}")

print("\nVQE |x> amplitudes (phase-aligned):")
for i, amp in enumerate(psi_aligned):
    print(f"  |{i:02b}>: {amp:.6f}")
